# Commodity Data Collector
### Prices (FRED) · News (SerpAPI) · FinBERT Embeddings & Sentiment

Covers **wheat**, **corn**, and **oil** from 2010-01-01 → 2026-03-31.

Output folder structure:
```
data/
  wheat/
    wheat_prices.csv
    wheat_news.csv
    wheat_news_embeddings.pt
    wheat_news_sentiment.csv
  corn/
    corn_prices.csv
    corn_news.csv
    corn_news_embeddings.pt
    corn_news_sentiment.csv
  oil/
    oil_prices.csv
    oil_news.csv
    oil_news_embeddings.pt
    oil_news_sentiment.csv
```

In [1]:
import subprocess, sys
_pkgs = ['fredapi', 'python-dotenv', 'google-search-results',
         'transformers', 'torch', 'scikit-learn', 'pandas',
         'numpy', 'tqdm', 'accelerate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _pkgs,
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('✓ Dependencies installed')

✓ Dependencies installed


---
## 1 · Configuration

In [2]:
import os, re, json, time, random, warnings, copy
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from fredapi import Fred
from serpapi import GoogleSearch
from tqdm import tqdm
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path=Path('.env'))

# ── API keys ──────────────────────────────────────────────────────────────────
FRED_API_KEY = os.getenv('FRED')
SERP_API_KEY = os.getenv('SERP_API')
assert FRED_API_KEY, 'FRED key missing from .env'
assert SERP_API_KEY, 'SERP_API key missing from .env'
print(f'✓ API keys loaded (FRED, SerpAPI)')

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = '2010-01-01'
END_DATE   = '2026-03-31'
YEARS      = list(range(2010, 2027))

# ── Commodity configuration ───────────────────────────────────────────────────
# No site: restriction — we cast wide and let dedup clean later.
# Each query is designed to pull a distinct news angle for that commodity.
COMMODITIES = [
    {
        'name': 'wheat',
        'fred_series': 'PWHEAMTUSDM',
        'fred_frequency': 'monthly',
        'serp_queries': [
            # Price & futures
            'wheat futures price market',
            'wheat price forecast commodity market',
            'CBOT wheat futures trading',
            # Supply & trade
            'wheat export supply global trade',
            'wheat import export shipment tons',
            'wheat production world supply demand',
            'wheat trade flows ports shipment',
            # Crop & weather
            'wheat crop harvest yield season',
            'wheat drought flood weather crop damage',
            'wheat planting acreage growing conditions',
            'winter wheat spring wheat harvest update',
            # Macro shocks
            'wheat Ukraine Russia war shortage',
            'wheat Black Sea grain deal corridor',
            'wheat sanctions embargo food crisis',
            'wheat food security hunger inflation',
            # Policy & reports
            'wheat USDA report wasde supply demand',
            'wheat tariff subsidy import export policy',
            'wheat government stockpile reserve release',
            # General
            'wheat market news analysis outlook',
            'wheat commodity news prices update',
        ],
    },
    {
        'name': 'corn',
        'fred_series': 'PMAIZMTUSDM',
        'fred_frequency': 'monthly',
        'serp_queries': [
            # Price & futures
            'corn futures price market',
            'corn price forecast commodity market',
            'CBOT corn futures trading',
            # Supply & trade
            'corn export supply global trade',
            'corn ethanol biofuel demand production',
            'corn import export shipment tons',
            'corn production world supply demand',
            # Crop & weather
            'corn crop harvest yield season',
            'corn drought flood weather crop damage',
            'corn planting acreage growing conditions',
            'corn silage feed grain harvest update',
            # Macro shocks
            'corn China US trade war imports',
            'corn food crisis shortage inflation',
            'corn feed grain livestock demand',
            # Policy & reports
            'corn USDA report wasde supply demand',
            'corn tariff subsidy import export policy',
            'corn ethanol renewable fuel standard policy',
            # General
            'corn market news analysis outlook',
            'corn commoxdity news prices update',
        ],
    },
    {
        'name': 'oil',
        'fred_series': 'DCOILWTICO',
        'fred_frequency': 'daily',
        'serp_queries': [
            # Price & futures
            'crude oil WTI futures price market',
            'Brent crude oil futures price market',
            'oil price forecast energy market',
            # Supply & production
            'OPEC oil production output decision',
            'oil supply production barrels per day',
            'shale oil US production rig count',
            'oil inventory stockpile drawdown EIA',
            # Demand & macro
            'crude oil demand China global economy',
            'oil demand forecast consumption growth',
            'oil refinery capacity utilization margin',
            # Geopolitical shocks
            'oil war sanctions Russia Iran geopolitical',
            'oil Venezuela Libya Nigeria supply disruption',
            'oil pipeline attack tanker strait',
            # Policy & reports
            'oil OPEC plus meeting output quota',
            'oil strategic petroleum reserve release',
            'oil energy transition carbon policy',
            # General
            'crude oil market news analysis outlook',
            'oil energy commodity news prices update',
        ],
    },
]

# ── SerpAPI settings ──────────────────────────────────────────────────────────
# No site: filter — broad Google News search, domain extracted from result URL.
# Paginate until last page (empty organic results) with no hard page cap.
REQUEST_BUDGET    = 10_000   # re-run to continue; done combos are always skipped
RESULTS_PER_PAGE  = 10
SERP_DELAY_MIN    = 0.8
SERP_DELAY_MAX    = 2.0

# ── Source domain extraction ──────────────────────────────────────────────────
def extract_domain(url: str) -> str:
    try:
        h = urlparse(url).netloc
        return h.replace('www.', '')
    except Exception:
        return ''

# ── Deduplication ─────────────────────────────────────────────────────────────
def normalize_title(t: str) -> str:
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', '', t.lower())).strip()

# ── FinBERT settings ──────────────────────────────────────────────────────────
FINBERT_MODEL  = 'ProsusAI/finbert'
EMBED_BATCH    = 8
PCA_DIM        = 16
MAX_TOKEN_LEN  = 512

# ── Data directory ────────────────────────────────────────────────────────────
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

DEVICE = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if getattr(torch.backends, 'mps', None)
          and torch.backends.mps.is_available() else torch.device('cpu'))

total_combos = sum(len(c['serp_queries']) * len(YEARS) * 12 for c in COMMODITIES)
print(f'✓ Device: {DEVICE}')
print(f'✓ Commodities: {[c["name"] for c in COMMODITIES]}')
print(f'✓ Total (query, year) combos: {total_combos:,}  (paginate to last page each)')
print(f'✓ SerpAPI budget per run: {REQUEST_BUDGET:,} requests (re-run to continue)')

✓ API keys loaded (FRED, SerpAPI)
✓ Device: mps
✓ Commodities: ['wheat']
✓ Total (query, year) combos: 340  (paginate to last page each)
✓ SerpAPI budget per run: 10,000 requests (re-run to continue)


<!-- ---
## 2 · Folder Structure -->

In [3]:
# Create commodity subfolders
for cfg in COMMODITIES:
    name = cfg['name']
    commodity_dir = DATA_DIR / name
    commodity_dir.mkdir(exist_ok=True)

print('✓ Folder structure created:')
for cfg in COMMODITIES:
    name = cfg['name']
    commodity_dir = DATA_DIR / name
    print(f'  data/{name}/')
    for f in ['prices', 'news', 'embeddings', 'sentiment']:
        if f == 'prices':
            print(f'    {name}_prices.csv')
        elif f == 'news':
            print(f'    {name}_news.csv')
        elif f == 'embeddings':
            print(f'    {name}_news_embeddings.pt')
        elif f == 'sentiment':
            print(f'    {name}_news_sentiment.csv')

✓ Folder structure created:
  data/wheat/
    wheat_prices.csv
    wheat_news.csv
    wheat_news_embeddings.pt
    wheat_news_sentiment.csv


---
## 3 · FRED Price Data

- **wheat / corn**: `PWHEAMTUSDM` / `PMAIZMTUSDM` — monthly global prices, resampled to business-daily.
- **oil**: `DCOILWTICO` — daily WTI spot price.

Files saved to `data/{commodity}/{commodity}_prices.csv`

In [4]:
fred = Fred(api_key=FRED_API_KEY)

def fetch_fred_prices(series_id: str, frequency: str,
                      start: str, end: str) -> pd.DataFrame:
    raw = fred.get_series(series_id, observation_start=start, observation_end=end)
    df = pd.DataFrame({'Price': raw}).reset_index()
    df.columns = ['Date', 'Price']
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.dropna(subset=['Price']).sort_values('Date')

    if frequency == 'monthly':
        df = (df.set_index('Date')
                .resample('B')
                .ffill()
                .reset_index())

    df = df[(df['Date'] >= START_DATE) & (df['Date'] <= END_DATE)]
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    return df.reset_index(drop=True)

print('Fetching FRED prices...\n')
for cfg in COMMODITIES:
    name = cfg['name']
    out_csv = DATA_DIR / name / f'{name}_prices.csv'

    if out_csv.exists():
        existing = pd.read_csv(out_csv)
        print(f'  {name}: already exists ({len(existing):,} rows) — skipping')
        continue

    df = fetch_fred_prices(cfg['fred_series'], cfg['fred_frequency'],
                           START_DATE, END_DATE)
    df.to_csv(out_csv, index=False)
    print(f'  {name}: {len(df):,} rows saved')

print()

Fetching FRED prices...

  wheat: already exists (4,216 rows) — skipping



---
## 4 · SerpAPI News Collection

**Strategy:** ~20 queries × 17 years × paginate to last page = maximum coverage per commodity.

| Dimension | Count | Detail |
|---|---|---|
| Queries | ~20 | futures/prices, exports/supply, crop/harvest/weather, macro shocks, policy/USDA, general — **no `site:` filter** |
| Years | 17 | 2010–2026 |
| Pages | until empty | 10 results/page, paginate until Google returns fewer than 10 |
| Source | from URL | domain parsed from each result link — no whitelist |

**Key changes from previous version:**
- Removed `site:` restriction — queries now hit all news sources (Reuters, Bloomberg, FT, Xinhua, AgriNewsWire, etc.)
- Uses `tbm=nws` (Google News tab) for much denser news indexing
- Paginate to the last page per (query, year) combo instead of stopping at page 3
- Checkpoint key: `{commodity}:q{idx}:{year}` — re-runs skip already-done combos
- Budget `REQUEST_BUDGET = 10_000` per run — re-run as many times as needed

> **Reset checkpoint:** run the reset cell below once to wipe the old checkpoint before starting.

In [5]:
# ── Reset checkpoint ──────────────────────────────────────────────────────────
# Run this cell ONCE if you want to wipe all progress and start fresh.
# New checkpoint key format: "{commodity}:q{idx}:{year}"  (no source — queries are now broad)

import json
_ckpt_path = Path('data') / 'serp_checkpoint.json'
_ckpt_path.write_text(json.dumps({'fetched': {}, 'total_requests': 0}, indent=2))
print('✓ Checkpoint reset — all combos will be re-fetched')

✓ Checkpoint reset — all combos will be re-fetched


In [6]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
CHECKPOINT_FILE = DATA_DIR / 'serp_checkpoint.json'

def load_checkpoint() -> dict:
    if CHECKPOINT_FILE.exists():
        return json.loads(CHECKPOINT_FILE.read_text())
    return {'fetched': {}, 'total_requests': 0}

def save_checkpoint(ckpt: dict):
    CHECKPOINT_FILE.write_text(json.dumps(ckpt, indent=2))

def ckpt_key(commodity: str, query_idx: int, year: int, month: int) -> str:
    # No source in key — queries are now broad (no site: filter)
    return f'{commodity}:q{query_idx}:{year}:{month:02d}'

def existing_seen_keys(csv_path: Path) -> set:
    """Load (normalized_title) set already saved to CSV for fast dedup."""
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path, usecols=['title'])
            return set(df['title'].fillna('').map(normalize_title).tolist())
        except Exception:
            pass
    return set()

# ── Date parser ───────────────────────────────────────────────────────────────
_MONTHS = {m: i+1 for i, m in enumerate(
    ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])}

def parse_serp_date(raw: str) -> str:
    if not raw:
        return ''
    if 'ago' in raw.lower():
        return datetime.utcnow().strftime('%Y-%m-%d')
    m = re.search(r'([A-Za-z]+)\s+(\d{1,2}),?\s+(\d{4})', raw)
    if m:
        mon_str, day, year = m.group(1)[:3].capitalize(), int(m.group(2)), int(m.group(3))
        mon = _MONTHS.get(mon_str, 0)
        if mon:
            return f'{year}-{mon:02d}-{day:02d}'
    m2 = re.search(r'(\d{4})-(\d{2})-(\d{2})', raw)
    if m2:
        return m2.group(0)
    return ''

# ── Core SerpAPI fetch — paginate to last page ────────────────────────────────
import calendar
def fetch_serp_combo(commodity: str, query: str, query_idx: int,
                     year: int, month: int, ckpt: dict) -> tuple[list[dict], int]:
    """
    Fetch one (commodity, query_idx, year) combo, paginating until no more results.
    No site: filter — captures all news sources; domain extracted from result URL.
    Returns (records, requests_used). requests_used == -1 means budget exhausted.
    """
    key = ckpt_key(commodity, query_idx, year, month)
    if key in ckpt['fetched']:
        return [], 0  # already done

    if ckpt['total_requests'] >= REQUEST_BUDGET:
        return [], -1

    records = []
    requests_used = 0
    page = 0

    while True:
        if ckpt['total_requests'] >= REQUEST_BUDGET:
            break

        params = {
            'q':       query,
            'api_key': SERP_API_KEY,
            'num':     RESULTS_PER_PAGE,
            'start':   page * RESULTS_PER_PAGE,
            'tbs':     f'cdr:1,cd_min:{month:02d}/01/{year},cd_max:{month:02d}/{calendar.monthrange(year, month)[1]}/{year}',
            'tbm':     'nws',   # Google News tab — much denser news coverage
        }

        try:
            time.sleep(random.uniform(SERP_DELAY_MIN, SERP_DELAY_MAX))
            result = GoogleSearch(params).get_dict()
            requests_used += 1
            ckpt['total_requests'] += 1

            organic = result.get('news_results') or result.get('organic_results', [])
            if not organic:
                break  # no more pages

            for r in organic:
                records.append({
                    'commodity':   commodity,
                    'title':       r.get('title', ''),
                    'date':        parse_serp_date(r.get('date', '')),
                    'source':      extract_domain(r.get('link', '')),
                    'description': r.get('snippet', ''),
                    'url':         r.get('link', ''),
                })

            # Stop if this page had fewer results than requested (last page)
            if len(organic) < RESULTS_PER_PAGE:
                break

            page += 1

        except Exception as e:
            print(f'    [warn] q{query_idx} {year}-{month:02d} page {page}: {e}')
            break

    ckpt['fetched'][key] = len(records)
    save_checkpoint(ckpt)
    return records, requests_used

print('✓ SerpAPI helpers ready  (broad search, paginate to last page, domain from URL)')
ckpt = load_checkpoint()
print(f'  Checkpoint: {ckpt["total_requests"]} requests used, '
      f'{len(ckpt["fetched"])} combos done')

✓ SerpAPI helpers ready  (broad search, paginate to last page, domain from URL)
  Checkpoint: 0 requests used, 0 combos done


In [7]:
# ── Main news collection loop ────────────────────────────────────────────────
# Iterates: commodity → query (20) → year (17) → page (until empty)
# No site: restriction — all news sources captured, domain parsed from URL.
# Deduplication by normalized_title — same headline never stored twice.
# Re-run as many times as needed; completed (commodity, query, year) combos are skipped.
ckpt = load_checkpoint()
budget_exhausted = False

total_combos = sum(len(c['serp_queries']) * len(YEARS) * 12 for c in COMMODITIES)
done_combos  = len(ckpt['fetched'])
print(f'SerpAPI budget: {ckpt["total_requests"]} / {REQUEST_BUDGET} requests used')
print(f'Combos done: {done_combos} / {total_combos}\n')

for cfg in COMMODITIES:
    if budget_exhausted:
        break

    name    = cfg['name']
    queries = cfg['serp_queries']
    out_csv = DATA_DIR / name / f'{name}_news.csv'

    # Seed seen-keys from whatever is already on disk
    seen_titles = existing_seen_keys(out_csv)
    existing_count = len(pd.read_csv(out_csv)) if out_csv.exists() else 0
    print(f'── {name.upper()} (on disk: {existing_count:,} articles) ──')
    commodity_new = 0

    for q_idx, query in enumerate(queries):
        if budget_exhausted:
            break
        query_new = 0

        for year in YEARS:
            for month in range(1, 13):
                if budget_exhausted:
                    break

                records, req_count = fetch_serp_combo(name, query, q_idx, year, month, ckpt)

            if req_count == -1:
                budget_exhausted = True
                break

            if not records:
                continue

            # Require a parsed date; deduplicate by normalized title
            new_records = []
            for r in records:
                if not r['date']:
                    continue
                norm = normalize_title(r['title'])
                if norm and norm not in seen_titles:
                    seen_titles.add(norm)
                    new_records.append(r)

            if new_records:
                df_new = pd.DataFrame(new_records)
                write_header = not out_csv.exists()
                df_new.to_csv(out_csv, mode='a', header=write_header, index=False)
                query_new     += len(new_records)
                commodity_new += len(new_records)

        if query_new:
            print(f'  q{q_idx+1:02d} "{query}": +{query_new}')

    total = len(pd.read_csv(out_csv)) if out_csv.exists() else 0
    print(f'  ✓ {name}: {total:,} articles total (+{commodity_new} this run)\n')

print(f'Requests used this session: {ckpt["total_requests"]} / {REQUEST_BUDGET}')
if budget_exhausted:
    print('⚠  Budget limit reached — re-run to continue (done combos are skipped)')

SerpAPI budget: 0 / 10000 requests used
Combos done: 0 / 340

── WHEAT (on disk: 383 articles) ──
  ✓ wheat: 383 articles total (+0 this run)

Requests used this session: 340 / 10000


---
## 5 · FinBERT Embeddings & Sentiment

For each commodity:
1. Load news CSV, clean text (title + description)
2. Extract 768-D `[CLS]` embeddings via FinBERT
3. PCA-reduce to 16-D
4. Score sentiment → positive / negative / neutral
5. Mean-pool per calendar day
6. Save:
   - `{commodity}_news_embeddings.pt` — `dict[date_str, Tensor(16,)]`
   - `{commodity}_news_sentiment.csv` — daily sentiment + article count

Skips if output files exist (delete to reprocess).

In [8]:
# ── Text cleaning ────────────────────────────────────────────────────────────
_BOILERPLATE_RE = re.compile(
    r'^(?:\*\s*)?(?:By\s+[A-Z][a-zA-Z\s\-\']+'
    r'[A-Z]{2,}[\w\s,]*\([^)]+\)\s*[-–—]\s*)?(?:\*\s*)?',
    re.MULTILINE,
)

def clean_text(row: pd.Series) -> str:
    title = str(row.get('title', '')).strip()
    desc = str(row.get('description', '')).strip()
    desc = _BOILERPLATE_RE.sub('', desc).strip()
    if desc.endswith('...'):
        desc = desc[:-3].strip()
    return f'{title}. {desc}' if desc else title

# ── Embedding extraction ──────────────────────────────────────────────────────
def extract_cls_embeddings(texts: list, tokenizer, model, device,
                           batch_size: int = EMBED_BATCH) -> torch.Tensor:
    all_emb = []
    for start in tqdm(range(0, len(texts), batch_size), desc='  Embeddings', leave=False):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_TOKEN_LEN, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        all_emb.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(all_emb, dim=0)

# ── PCA reduction ─────────────────────────────────────────────────────────────
def apply_pca(emb: torch.Tensor, n: int = PCA_DIM) -> torch.Tensor:
    pca = PCA(n_components=n)
    reduced = pca.fit_transform(emb.numpy())
    var = pca.explained_variance_ratio_.sum()
    print(f'  PCA {emb.shape[1]}-D → {n}-D (var: {var:.3f})')
    return torch.from_numpy(reduced).float()

# ── Sentiment extraction ──────────────────────────────────────────────────────
def extract_sentiment(texts: list, tokenizer, sent_model, device,
                      batch_size: int = EMBED_BATCH) -> pd.DataFrame:
    rows = []
    for start in tqdm(range(0, len(texts), batch_size), desc='  Sentiment', leave=False):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_TOKEN_LEN, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            probs = torch.softmax(sent_model(**enc).logits, dim=-1).cpu()
        for row in probs:
            pos, neg, neu = row[0].item(), row[1].item(), row[2].item()
            rows.append({'sentiment_pos': pos, 'sentiment_neg': neg,
                         'sentiment_neu': neu, 'sentiment_score': pos - neg})
    return pd.DataFrame(rows)

# ── Daily aggregation ─────────────────────────────────────────────────────────
def aggregate_daily(df: pd.DataFrame, emb: torch.Tensor) -> dict:
    dates = df['date_day'].tolist()
    idx_map = {}
    for i, d in enumerate(dates):
        idx_map.setdefault(d, []).append(i)
    return {day: torch.mean(emb[idxs], dim=0)
            for day, idxs in sorted(idx_map.items())}

print('✓ FinBERT helpers ready')

✓ FinBERT helpers ready


In [9]:
print('Processing FinBERT embeddings & sentiment...\n')

for cfg in COMMODITIES:
    name = cfg['name']
    news_csv  = DATA_DIR / name / f'{name}_news.csv'
    emb_path  = DATA_DIR / name / f'{name}_news_embeddings.pt'
    sent_path = DATA_DIR / name / f'{name}_news_sentiment.csv'

    if not news_csv.exists():
        print(f'{name}: news CSV not found — run §4 first')
        continue

    if emb_path.exists() and sent_path.exists():
        print(f'{name}: already processed (delete files to reprocess)')
        continue

    print(f'{name.upper()}:')

    # Load, deduplicate by (source, normalized_title), filter dates
    df = pd.read_csv(news_csv)
    before = len(df)
    df['_key'] = df['source'].fillna('') + '||' + df['title'].fillna('').map(normalize_title)
    df = df.drop_duplicates(subset='_key').drop(columns='_key')
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    df['date_day'] = df['date'].dt.strftime('%Y-%m-%d')
    df = df[(df['date_day'] >= START_DATE) & (df['date_day'] <= END_DATE)]
    df['clean_text'] = df.apply(clean_text, axis=1)
    df = df.reset_index(drop=True)
    print(f'  {before:,} raw → {len(df):,} after dedup & date filter '
          f'({df["date_day"].nunique():,} unique days)')

    texts = df['clean_text'].tolist()

    # Load FinBERT base model
    tokenizer  = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    base_model = AutoModel.from_pretrained(FINBERT_MODEL).to(DEVICE).eval()

    emb_768 = extract_cls_embeddings(texts, tokenizer, base_model, DEVICE, EMBED_BATCH)
    del base_model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    emb_16 = apply_pca(emb_768, PCA_DIM)

    # Sentiment
    sent_model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL).to(DEVICE).eval()
    sent_df = extract_sentiment(texts, tokenizer, sent_model, DEVICE, EMBED_BATCH)
    del sent_model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    df = pd.concat([df.reset_index(drop=True), sent_df], axis=1)

    # Daily aggregation
    daily_emb = aggregate_daily(df, emb_16)
    daily_sent = (df.groupby('date_day')[
                      ['sentiment_pos', 'sentiment_neg', 'sentiment_neu', 'sentiment_score']]
                    .mean()
                    .reset_index()
                    .rename(columns={'date_day': 'date'}))
    daily_sent['article_count'] = df.groupby('date_day').size().values
    daily_sent = daily_sent.sort_values('date').reset_index(drop=True)

    torch.save(daily_emb, emb_path)
    daily_sent.to_csv(sent_path, index=False)
    print(f'  Saved: {len(daily_emb)} embedding days, {len(daily_sent)} sentiment rows')
    print()

Processing FinBERT embeddings & sentiment...

wheat: already processed (delete files to reprocess)


---
## 6 · Summary

In [10]:
print('=' * 70)
print('COMMODITY DATA COLLECTION — FINAL SUMMARY')
print('=' * 70)
print()

ckpt_final = load_checkpoint()
print(f'SerpAPI requests used: {ckpt_final["total_requests"]} / {REQUEST_BUDGET}')
print()

for cfg in COMMODITIES:
    name = cfg['name']
    price_csv = DATA_DIR / name / f'{name}_prices.csv'
    news_csv = DATA_DIR / name / f'{name}_news.csv'
    emb_path = DATA_DIR / name / f'{name}_news_embeddings.pt'
    sent_path = DATA_DIR / name / f'{name}_news_sentiment.csv'

    n_prices = len(pd.read_csv(price_csv)) if price_csv.exists() else 0
    n_news = len(pd.read_csv(news_csv)) if news_csv.exists() else 0
    n_emb = len(torch.load(emb_path, weights_only=False)) if emb_path.exists() else 0
    n_sent = len(pd.read_csv(sent_path)) if sent_path.exists() else 0

    print(f'{name.upper()}')
    print(f'  Prices: {n_prices:,} rows')
    print(f'  News:   {n_news:,} articles  |  Embeddings: {n_emb:,} days  |  Sentiment: {n_sent:,} days')
    print()

print('Files in data/:')
for cfg in COMMODITIES:
    name = cfg['name']
    dir_path = DATA_DIR / name
    if dir_path.exists():
        for f in sorted(dir_path.glob('*')):
            size_kb = f.stat().st_size / 1024
            print(f'  {str(f.relative_to(DATA_DIR)):40s}  {size_kb:8.1f} KB')

COMMODITY DATA COLLECTION — FINAL SUMMARY

SerpAPI requests used: 340 / 10000

WHEAT
  Prices: 4,216 rows
  News:   383 articles  |  Embeddings: 335 days  |  Sentiment: 335 days

Files in data/:
  wheat/wheat_news.csv                         127.8 KB
  wheat/wheat_news_embeddings.pt               118.3 KB
  wheat/wheat_news_sentiment.csv                30.3 KB
  wheat/wheat_prices.csv                       115.4 KB
